<a href="https://colab.research.google.com/github/jimqu06/Triton-NeuroTech/blob/main/Workday3_FA_CNN_CIFAR10_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FA inside a CNN: backprop vs. feedback alignment on CIFAR-10
**Triton NeuroTech research project · Workday 3**

Last week FA trained an MLP on MNIST. Today: the same idea inside **convolutional** layers, on harder images (CIFAR-10: 32 × 32 color photos, 10 classes). We train the **same CNN** with each rule and **save checkpoints**, which we'll compare to human EEG in Workday 4.

**Use a GPU:** in Colab, *Runtime > Change runtime type > T4 GPU*.

## Settings

**Working in groups:** each group sets its own `SEED` and FA learning rate (the leads will assign them), trains both rules, and posts its `RESULTS` line. Together the groups show how much results vary between seeds, and what the learning rate does to FA.

In [ ]:
GROUP = 'your names here'
EPOCHS = 10                      # training passes over the 50,000 training images
LR = {'bp': 1e-3, 'fa': 1e-4}     # learning rate per rule (Adam); some groups change FA's
SEED = 0                         # each group gets its own seed; the project uses 5 seeds per rule
SAVE_AT = [0, 1, 2, 3, 5, 10]    # epochs to save checkpoints at (0 = untrained, our baseline)
CHANNELS, HIDDEN = (32, 64, 128), 256

## Setup and data

In [ ]:
import math, time, os, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.grad import conv2d_input, conv2d_weight
import matplotlib.pyplot as plt
from torchvision import datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on', device, '' if device.type == 'cuda' else '  <- switch to a GPU runtime for speed')

In [ ]:
train_set = datasets.CIFAR10('data', train=True, download=True)
test_set = datasets.CIFAR10('data', train=False, download=True)
CLASSES = train_set.classes

def to_tensors(ds):   # keep raw uint8 pixels on the device: (N, 3, 32, 32)
    x = torch.tensor(ds.data).permute(0, 3, 1, 2).contiguous().to(device)
    return x, torch.tensor(ds.targets).to(device)

x_train, y_train = to_tensors(train_set)
x_test, y_test = to_tensors(test_set)

MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1, 3, 1, 1)

def normalize(xb):    # uint8 -> standardized float, per color channel
    return (xb.float() / 255 - MEAN) / STD

print('train', tuple(x_train.shape), ' test', tuple(x_test.shape))
fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, cls in zip(axes, range(10)):
    i = (y_train == cls).nonzero()[0, 0]
    ax.imshow(x_train[i].permute(1, 2, 0).cpu()); ax.set_title(CLASSES[cls], fontsize=8); ax.axis('off')
plt.show()

## FA layers
`FALinear` is last week's layer. `FAConv2d` is new, and it's **Proof 1** in code:
- The forward pass is a normal convolution.
- The backward pass sends the error down with a **transposed convolution**: backprop uses the layer's own kernel W, and FA uses a fixed random kernel B.
- The weight gradient (error × input) is the same for both rules.

In [ ]:
class FALinearFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, W, b, B):
        ctx.save_for_backward(x, W, B)
        return x @ W.T + b

    @staticmethod
    def backward(ctx, grad_output):
        x, W, B = ctx.saved_tensors
        return grad_output @ B, grad_output.T @ x, grad_output.sum(0), None


class FALinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.register_buffer('B', torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))

    def forward(self, x):
        return FALinearFunction.apply(x, self.weight, self.bias, self.B)

In [ ]:
class FAConv2dFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, W, b, B, padding):
        ctx.save_for_backward(x, W, B)
        ctx.padding = padding
        return F.conv2d(x, W, b, padding=padding)

    @staticmethod
    def backward(ctx, grad_output):
        x, W, B = ctx.saved_tensors
        p = ctx.padding
        # Error to send down to the layer below (same shape as x).
        # Backprop would write:   grad_input = conv2d_input(x.shape, W, grad_output, padding=p)
        # ---------------- YOUR CODE: one line ----------------
        grad_input = None   # <- replace None with FA's version
        # ------------------------------------------------------
        if grad_input is None:
            raise NotImplementedError('Fill in grad_input in FAConv2dFunction.backward (see the hint above it).')

        # How this layer's own kernel should change: slide the error over the input (Proof 1, Claim 2).
        grad_W = conv2d_weight(x, W.shape, grad_output, padding=p)      # same as backprop
        grad_b = grad_output.sum(dim=(0, 2, 3))
        return grad_input, grad_W, grad_b, None, None                   # B and padding get no gradient


class FAConv2d(nn.Module):
    """Drop-in replacement for nn.Conv2d that learns with feedback alignment."""

    def __init__(self, in_channels, out_channels, kernel_size, padding=0):
        super().__init__()
        self.padding = padding
        shape = (out_channels, in_channels, kernel_size, kernel_size)
        self.weight = nn.Parameter(torch.empty(shape))      # W: learned
        self.bias = nn.Parameter(torch.zeros(out_channels))
        self.register_buffer('B', torch.empty(shape))        # B: random, fixed forever
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))

    def forward(self, x):
        return FAConv2dFunction.apply(x, self.weight, self.bias, self.B, self.padding)

In [ ]:
def check_fa_conv():
    torch.manual_seed(0)
    x = torch.randn(4, 3, 8, 8, requires_grad=True)
    W, b, B = torch.randn(5, 3, 3, 3), torch.randn(5), torch.randn(5, 3, 3, 3)
    g = torch.randn(4, 5, 8, 8)                        # pretend error from the layer above
    try:
        FAConv2dFunction.apply(x, W, b, B, 1).backward(g)
    except NotImplementedError:
        print('Not filled in yet: replace None in FAConv2dFunction.backward.'); return False
    except RuntimeError as e:
        print('The shapes do not line up. grad_input must have the same shape as x. Did you keep padding=p?'); return False
    if torch.allclose(x.grad, F.conv_transpose2d(g, W, padding=1), atol=1e-5):
        print('Close! That is backprop: the error went back through W. FA uses B.'); return False
    if not torch.allclose(x.grad, F.conv_transpose2d(g, B, padding=1), atol=1e-5):
        print('Not quite. Hint: copy the backprop line from the comment and change one letter.'); return False
    # With B = W, FA must reproduce backprop exactly
    ref = nn.Conv2d(3, 5, 3, padding=1)
    x1 = torch.randn(2, 3, 8, 8, requires_grad=True); x2 = x1.detach().clone().requires_grad_(True)
    W1 = ref.weight.detach().clone().requires_grad_(True)
    FAConv2dFunction.apply(x1, W1, ref.bias.detach(), ref.weight.detach(), 1).pow(2).sum().backward()
    ref(x2).pow(2).sum().backward()
    same = torch.allclose(x1.grad, x2.grad, atol=1e-5) and torch.allclose(W1.grad, ref.weight.grad, atol=1e-5)
    print('PASS: the error goes back through a transposed convolution with the random kernel B (Proof 1).')
    print('      and with B = W it matches backprop exactly:', same)
    return same

fa_ready = check_fa_conv()

## The CNN: identical for both rules
Three conv layers (3 × 3, padding 1), each followed by ReLU and 2 × 2 max pooling, then two fully connected layers. There's no batch norm, on purpose: it's the source of the 2026 retraction, and it interacts with FA. The **only** difference between the two models is which layer classes they use.

In [ ]:
class CNN(nn.Module):
    def __init__(self, rule='bp', channels=CHANNELS, hidden=HIDDEN):
        super().__init__()
        if rule == 'bp':
            Conv = lambda i, o: nn.Conv2d(i, o, 3, padding=1)
            Linear = nn.Linear
        else:
            Conv = lambda i, o: FAConv2d(i, o, 3, padding=1)
            Linear = FALinear
        c1, c2, c3 = channels
        self.conv1, self.conv2, self.conv3 = Conv(3, c1), Conv(c1, c2), Conv(c2, c3)   # 32 -> 16 -> 8 -> 4 pixels
        self.fc1 = Linear(c3 * 4 * 4, hidden)
        self.fc2 = Linear(hidden, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.max_pool2d(F.relu(self.conv3(x)), 2)
        return self.fc2(F.relu(self.fc1(x.flatten(1))))


for rule in ['bp', 'fa']:
    m = CNN(rule)
    print(rule.upper(), 'learnable parameters:', sum(p.numel() for p in m.parameters()))

## Training, with timing and checkpoints
Each checkpoint stores the weights, the random feedback matrices B (they're part of an FA model), the epoch and the test accuracy. Checkpoints at several accuracies are what the **matched-accuracy** analysis needs (Proof 7).

In [ ]:
os.makedirs('checkpoints', exist_ok=True)


@torch.no_grad()
def accuracy(model):
    model.eval()
    correct = sum((model(normalize(x_test[i:i + 1000])).argmax(1) == y_test[i:i + 1000]).sum().item()
                  for i in range(0, len(x_test), 1000))
    return correct / len(x_test)


def save_checkpoint(model, rule, seed, epoch, acc):
    path = f'checkpoints/{rule}_seed{seed}_epoch{epoch:02d}.pt'
    torch.save({'state_dict': model.state_dict(), 'rule': rule, 'seed': seed, 'epoch': epoch,
                'test_acc': acc, 'lr': LR[rule], 'channels': CHANNELS, 'hidden': HIDDEN}, path)


def load_checkpoint(path):
    ck = torch.load(path, map_location=device)
    model = CNN(ck['rule'], ck['channels'], ck['hidden']).to(device)
    model.load_state_dict(ck['state_dict'])
    return model.eval(), ck


def train_model(rule, epochs=EPOCHS, seed=SEED, batch_size=128):
    torch.manual_seed(seed)
    model = CNN(rule).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR[rule])
    history = {'epoch': [0], 'test_acc': [accuracy(model)], 'seconds': [0.0]}
    if 0 in SAVE_AT:
        save_checkpoint(model, rule, seed, 0, history['test_acc'][0])
    for epoch in range(1, epochs + 1):
        model.train()
        start = time.time()
        order = torch.randperm(len(x_train), device=device)
        for i in range(0, len(x_train), batch_size):
            idx = order[i:i + batch_size]
            xb = x_train[idx]
            flip = torch.rand(len(idx), device=device) < 0.5               # random left-right flips
            xb = torch.where(flip[:, None, None, None], xb.flip(3), xb)
            loss = F.cross_entropy(model(normalize(xb)), y_train[idx])
            opt.zero_grad()
            loss.backward()                                                # BP or FA happens here
            opt.step()
        if device.type == 'cuda':
            torch.cuda.synchronize()
        seconds = time.time() - start
        acc = accuracy(model)
        history['epoch'].append(epoch); history['test_acc'].append(acc); history['seconds'].append(seconds)
        print(f'{rule.upper()} epoch {epoch:2d}: test accuracy {acc:.2%}   ({seconds:.1f} s)')
        if epoch in SAVE_AT:
            save_checkpoint(model, rule, seed, epoch, acc)
    json.dump(history, open(f'checkpoints/{rule}_seed{seed}_history.json', 'w'))
    return model, history

In [ ]:
assert fa_ready, 'Get a PASS from check_fa_conv() first.'
bp_model, bp_hist = train_model('bp')

In [ ]:
fa_model, fa_hist = train_model('fa')

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(bp_hist['epoch'], [a * 100 for a in bp_hist['test_acc']], 'o-', label='Backprop')
ax.plot(fa_hist['epoch'], [a * 100 for a in fa_hist['test_acc']], 'o-', label='Feedback alignment')
ax.axhline(10, ls='--', c='gray', lw=1); ax.text(EPOCHS, 11, 'chance (10%)', ha='right', color='gray', fontsize=9)
ax.set_xlabel('epoch'); ax.set_ylabel('test accuracy (%)'); ax.set_title('Same CNN, two learning rules (CIFAR-10)')
ax.legend(); plt.tight_layout(); plt.show()

for name, h in [('BP', bp_hist), ('FA', fa_hist)]:
    per_epoch = sum(h['seconds'][1:]) / (len(h['seconds']) - 1)
    print(f"{name}: final test accuracy {h['test_acc'][-1]:.2%}   average {per_epoch:.1f} s per epoch")
print('Saved checkpoints:', sorted(os.listdir('checkpoints')))
sec = lambda h: sum(h['seconds'][1:]) / (len(h['seconds']) - 1)
print(f"RESULTS | {GROUP} | seed {SEED} | lr BP {LR['bp']:g}, FA {LR['fa']:g} | BP {bp_hist['test_acc'][-1]:.2%} | "
      f"FA {fa_hist['test_acc'][-1]:.2%} | {sec(bp_hist):.1f} / {sec(fa_hist):.1f} s per epoch")

**Post your `RESULTS` line in the Discord thread.** Once every group has posted, compare:
1. Groups with the same settings but different seeds: how much do BP and FA accuracy vary?
2. Groups with different FA learning rates: which one trains best, and is any of them unstable?
3. Is the BP–FA gap bigger than the seed-to-seed spread?

## Bonus: how well does each FA layer align?
Last week we measured the angle between FA's error signal and backprop's for one layer of an MLP. Here we measure it for **every** layer that sends an error down, at the start of training and at the end. Under 90° means FA's updates still point downhill (Proof 1's "why FA struggles in CNNs" is a claim about this angle).

In [ ]:
def layer_angles(model, n=500):
    """Angle (degrees) between what each FA layer sends down and what backprop would send."""
    store, hooks = {}, []
    for name, mod in model.named_children():
        if isinstance(mod, (FAConv2d, FALinear)):
            def hook(mod, inp, out, name=name):
                store[name] = {'x': inp[0].detach()}
                out.register_hook(lambda g, name=name: store[name].__setitem__('g', g.detach()))
            hooks.append(mod.register_forward_hook(hook))
    model.eval()
    F.cross_entropy(model(normalize(x_test[:n])), y_test[:n]).backward()
    for h in hooks:
        h.remove()
    model.zero_grad()
    angles = {}
    for name, mod in model.named_children():
        if name == 'conv1' or name not in store:      # conv1 sends its error to the image, which doesn't learn
            continue
        x, g = store[name]['x'], store[name]['g']
        if isinstance(mod, FAConv2d):
            fa = conv2d_input(x.shape, mod.B, g, padding=mod.padding)
            bp = conv2d_input(x.shape, mod.weight.detach(), g, padding=mod.padding)
        else:
            fa, bp = g @ mod.B, g @ mod.weight.detach()
        cos = (fa.flatten() @ bp.flatten()) / (fa.norm() * bp.norm())
        angles[name] = math.degrees(math.acos(cos.clamp(-1, 1).item()))
    return angles

start_model, _ = load_checkpoint(f'checkpoints/fa_seed{SEED}_epoch00.pt')
a0, a1 = layer_angles(start_model), layer_angles(fa_model)
names = list(a0)
xs = list(range(len(names)))
plt.figure(figsize=(6.5, 3.6))
for i, n in enumerate(names):
    plt.plot([i, i], [a0[n], a1[n]], c='gray', lw=1.5, zorder=1)
plt.scatter(xs, [a0[n] for n in names], s=60, c='lightgray', edgecolors='gray', zorder=2, label='start of training')
plt.scatter(xs, [a1[n] for n in names], s=60, zorder=3, label='end of training')
plt.axhline(90, ls='--', c='gray', lw=1)
plt.text(len(names) - 0.6, 90.6, 'unrelated (90°)', color='gray', fontsize=9, ha='right')
plt.xticks(xs, [f'{n}\n(sends to layer below)' for n in names], fontsize=8)
plt.xlim(-0.5, len(names) - 0.5); plt.ylim(min(a1.values()) - 5, 96)
plt.ylabel('angle to backprop (degrees)'); plt.title('Lower = better aligned with backprop'); plt.legend(loc='lower left')
plt.tight_layout(); plt.show()
for n in names:
    print(f'{n:6s} start {a0[n]:5.1f}°   end {a1[n]:5.1f}°')

## Keep the checkpoints
Colab deletes files when the runtime resets. Either download the zip, or save to Google Drive.

In [ ]:
import shutil
shutil.make_archive('checkpoints', 'zip', 'checkpoints')
print('Made checkpoints.zip. In Colab, download it from the Files panel (folder icon on the left).')

# Or copy to Google Drive (uncomment in Colab):
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copytree('checkpoints', '/content/drive/MyDrive/tnt_checkpoints', dirs_exist_ok=True)

## Wrap-up
1. How big is the BP–FA accuracy gap here compared with last week's MLP on MNIST? Why might conv layers make FA's job harder?
2. In Workday 4 we compare these checkpoints to human EEG. If FA's representations match the brain *worse*, how would we know whether that's the learning rule or just the lower accuracy? (Proof 7)